# 알약 탐지·분류 통합 추론

학습된 탐지기와 분류기 `best.pt`를 불러와 원본 이미지에서 알약 위치와 세부 품목을 순서대로 예측한다. 두 모델의 학습은 기존 개별 노트북에서 수행하고, 이 노트북은 pill recognition 추론과 결과 저장에 사용한다.

In [ ]:
import sys
from pathlib import Path

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd

from configs.pill_recognition import PillRecognitionConfig
from data_loading.factory import load_data_factory
from pipeline import (
    plot_pill_predictions,
)
from pipeline.pill_recognition import recognize_pills_from_config

In [ ]:
output_root = project_root / "outputs"
detector_run_name = "ssdlite320_baseline"
classifier_run_name = "resnet50_baseline"
inference_factory_path = "data_loading.recognition:build_pill_recognition_inputs"
input_dir = project_root / "data" / "inference"
inference_batch_size = 4
score_threshold = 0.25

detector_run_dir = output_root / detector_run_name
classifier_run_dir = output_root / classifier_run_name
detector_checkpoint_path = detector_run_dir / "best.pt"
classifier_checkpoint_path = classifier_run_dir / "best.pt"
detector_label_map_path = detector_run_dir / "label_map.json"
classifier_class_map_path = classifier_run_dir / "class_map.json"
result_dir = output_root / "pill_recognition"
print("device=auto")

## 추론 데이터 연결

`input_dir`에 추론할 이미지를 넣으면 기본 factory가 재귀적으로 읽는다. 외부 factory를 사용할 경우 `factory(config=config, batch_size=inference_batch_size)` 호출을 지원하고 `images: list[Tensor]`와 `meta: list[dict]`를 반환해야 한다.

In [ ]:
config = PillRecognitionConfig(
    inference_factory_path=inference_factory_path,
    input_dir=input_dir,
    detector_checkpoint_path=detector_checkpoint_path,
    classifier_checkpoint_path=classifier_checkpoint_path,
    detector_label_map_path=detector_label_map_path,
    classifier_class_map_path=classifier_class_map_path,
    output_dir=result_dir,
    detector_name=detector_run_name,
    batch_size=inference_batch_size,
    score_threshold=score_threshold,
    show_progress=True,
    device="auto",
)

## Best checkpoint 로드

두 checkpoint는 현재 label map 및 class map과 일치해야 한다. 모델은 같은 CUDA, MPS 또는 CPU 장치에서 순서대로 실행된다.

In [ ]:
result = recognize_pills_from_config(config)
print("통합 추론과 artifact 저장 완료")

## Pill recognition 추론

탐지 bbox를 이미지 경계로 보정한 뒤 crop한다. 가로세로 비율을 유지하기 위한 여백은 ImageNet RGB 평균에 해당하는 `(124, 116, 104)`로 채운다. 따라서 ImageNet 정규화 후 padding 영역은 거의 0이 되어 검정이나 흰색 여백보다 중립적인 입력이 된다. 최종 점수는 `detector_score × classifier_score`로 계산한다.

In [ ]:
predictions_path = result_dir / "pill_predictions.json"
errors_path = result_dir / "pill_recognition_errors.json"
print(f"predictions={len(result.predictions)}, detector_errors={len(result.detector_errors)}, classifier_errors={len(result.classifier_errors)}")
print(f"saved: {predictions_path}")
print(f"saved: {errors_path}")

In [ ]:
prediction_table = pd.DataFrame(
    [prediction.to_dict() for prediction in result.predictions]
)
display(prediction_table.head(20))

error_table = pd.DataFrame(
    [
        {"stage": "detector", **error.to_dict()}
        for error in result.detector_errors
    ]
    + [
        {"stage": "classifier", **error.to_dict()}
        for error in result.classifier_errors
    ]
)
display(error_table.head(20))

## 예측 시각화

첫 번째 추론 이미지에 최종 품목명, bbox와 결합 점수를 표시하고 PNG로 저장한다.

In [ ]:
inference_factory = load_data_factory(
    config.inference_factory_path,
    setting_name="inference_factory_path",
)
visualization_loader = inference_factory(
    config=config,
    batch_size=config.batch_size,
)
first_batch = next(iter(visualization_loader))
first_image = first_batch["images"][0]
first_image_id = str(first_batch["meta"][0]["image_id"])
first_predictions = [
    prediction
    for prediction in result.predictions
    if prediction.image_id == first_image_id
]
figure = plot_pill_predictions(
    first_image,
    first_predictions,
    title=f"Pill recognition image_id={first_image_id}",
)
visualization_dir = result_dir / "visualizations"
visualization_dir.mkdir(parents=True, exist_ok=True)
figure.savefig(
    visualization_dir / f"image-{first_image_id}.png",
    bbox_inches="tight",
)
figure